# 🚀 Xyether Strong Series (v2.5 / v3) Training Suite
### Complete 1:1 Reproduction & Training Engine for Google Colab Pro (Nvidia L4 / A100)

This notebook trains an anime super-resolution model cloning the **Xyether architecture**:
- **Base:** Global Nearest Residual (Overshoot / Ringing = 0.00)
- **Body:** 18 Convolutions + 17 Learnable PReLUs (600,652 parameters)
- **Pre-emphasis:** High-Pass edge derivative filter
- **Curriculum:** Phase 1 (Fidelity Base) &rarr; Phase 2 (Sharp Polish via D-UNet & FFLoss)
- **Checkpoint Source:** Continued from `net_g_89000.pth` with Zero-Init Reconstruction Head

In [ ]:
# @title 🔧 1. Environment Setup & Google Drive Mount
# @markdown Mount Google Drive to automatically persist checkpoints across sessions.

import os, sys, subprocess
from pathlib import Path
from IPython.display import display, HTML, clear_output

def print_status(msg, color="#4CAF50"):
    display(HTML(f"<pre style='color:{color}; font-weight:bold; font-family:Consolas, monospace;'>[SETUP] {msg}</pre>"))

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/Xyether_Training')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print_status(f"Google Drive connected! Checkpoints will save to: {DRIVE_DIR}")
except Exception as e:
    print_status("Google Drive not mounted. Checkpoints will be stored locally in /content/output.", "#FF9800")

# 2. Check GPU & Enable TensorFloat-32 for L4 / A100
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print_status(f"GPU Ready: {gpu_name} ({vram:.1f} GB VRAM) - TF32 Activated!", "#00E676")
else:
    print_status("No GPU detected! Please go to Runtime -> Change runtime type -> Select T4/L4/A100 GPU.", "#FF5252")

# 3. Install core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-msssim", "kornia", "onnx", "aria2"], check=False)
print_status("All dependencies ready! ✔️")

In [ ]:
# @title 📦 2. Download Training Engine & Source Checkpoint
# @markdown Pulls `net_g_89000.pth` and performs weight surgery (Zero-Init Head + Nearest Residual).

import urllib.request, shutil
from pathlib import Path

WORKDIR = Path("/content/Xyether_Engine")
WORKDIR.mkdir(parents=True, exist_ok=True)
sys.path.append(str(WORKDIR))

# Copy engine modules from local directory or download
CKPT_URL = "https://huggingface.co/Thanawanit/Kaggle-Backup/resolve/main/checkpoints/Phase4_XyetherKiller_V11/net_g_89000.pth"
SOURCE_CKPT = WORKDIR / "net_g_89000.pth"
BASE_CKPT = WORKDIR / "xyether_transplanted_base.pth"

if not SOURCE_CKPT.exists():
    print_status("Downloading net_g_89000.pth (4.62 MB)...", "#2196F3")
    subprocess.run(["aria2c", "-q", "-c", "-x", "8", "-s", "8", "-o", "net_g_89000.pth", CKPT_URL], cwd=str(WORKDIR))

# Perform Weight Surgery
import torch
from collections import OrderedDict

data = torch.load(str(SOURCE_CKPT), map_location='cpu')
sd = data.get('params_ema', data.get('params', data))

new_sd = OrderedDict()
for k, v in sd.items():
    clean_k = k.replace("module.", "")
    if "34" in clean_k:  # Conv 18 Reconstruction Head
        new_sd[clean_k] = torch.zeros_like(v)
    else:
        new_sd[clean_k] = v.clone()

torch.save({'params': new_sd, 'params_ema': new_sd}, str(BASE_CKPT))
print_status(f"Weight surgery complete! Base saved at: {BASE_CKPT}", "#00E676")

In [ ]:
# @title 🚀 3. Phase 1: Train Fidelity & Denoise Base
# @markdown Run Phase 1 to teach the 18 layers anti-aliasing residual and 91.5% noise elimination.

DATASET_PATH = "/content/dataset" # @param {type:"string"}
TOTAL_ITERS_P1 = 12000 # @param {type:"integer"}
BATCH_SIZE_P1 = 16 # @param {type:"integer"}

# Launch Phase 1
cmd = [
    sys.executable, "train_phase1.py",
    "--data_dir", DATASET_PATH,
    "--pretrained_path", str(BASE_CKPT),
    "--output_dir", "/content/drive/MyDrive/Xyether_Training/Phase1" if Path("/content/drive/MyDrive").exists() else "/content/experiments/phase1",
    "--batch_size", str(BATCH_SIZE_P1),
    "--total_iters", str(TOTAL_ITERS_P1),
    "--lr", "1e-4",
    "--eval_every", "1000",
    "--save_every", "2000"
]

print_status(f"Starting Phase 1 Training ({TOTAL_ITERS_P1} iterations)...", "#00E676")
subprocess.run(cmd)

In [ ]:
# @title ✨ 4. Phase 2: Train Sharp Polish (FFLoss + D-UNet)
# @markdown Refines Conv 17 and Conv 18 with Focal Frequency Loss and Spectral-Norm D-UNet for razor-sharp anime lines.

P1_DIR = Path("/content/drive/MyDrive/Xyether_Training/Phase1") if Path("/content/drive/MyDrive").exists() else Path("/content/experiments/phase1")
p1_ckpts = sorted(list(P1_DIR.glob("xyether_phase1_step_*.pth")), key=lambda p: int(p.stem.split('_')[-1]) if p.stem.split('_')[-1].isdigit() else 0)

if len(p1_ckpts) == 0:
    raise FileNotFoundError("No Phase 1 checkpoints found! Please run Cell 3 first.")

LATEST_P1_CKPT = p1_ckpts[-1]
print_status(f"Loaded Phase 1 Checkpoint: {LATEST_P1_CKPT}", "#2196F3")

TOTAL_ITERS_P2 = 6000 # @param {type:"integer"}
BATCH_SIZE_P2 = 12 # @param {type:"integer"}

cmd_p2 = [
    sys.executable, "train_phase2.py",
    "--data_dir", DATASET_PATH,
    "--pretrained_g", str(LATEST_P1_CKPT),
    "--output_dir", "/content/drive/MyDrive/Xyether_Training/Phase2" if Path("/content/drive/MyDrive").exists() else "/content/experiments/phase2",
    "--batch_size", str(BATCH_SIZE_P2),
    "--total_iters", str(TOTAL_ITERS_P2),
    "--lr_g", "3e-5",
    "--lr_d", "3e-5",
    "--ffl_weight", "0.05",
    "--gan_weight", "0.10",
    "--eval_every", "1000",
    "--save_every", "2000"
]

print_status(f"Starting Phase 2 Polish ({TOTAL_ITERS_P2} iterations)...", "#00E676")
subprocess.run(cmd_p2)

In [ ]:
# @title 🔬 5. Final Benchmark & Visual Verification
# @markdown Run the complete benchmark suite to verify Overshoot = 0.00, Denoise > 90%, and Line Depth <= 8.0.

from benchmark.xyether_evaluator import XyetherEvaluator
from archs.xyether_compact import XyetherCompactNet

P2_DIR = Path("/content/drive/MyDrive/Xyether_Training/Phase2") if Path("/content/drive/MyDrive").exists() else Path("/content/experiments/phase2")
final_model_path = P2_DIR / "Xyether_Strong_v3_Final.pth"

if final_model_path.exists():
    model = XyetherCompactNet().to(device)
    ckpt = torch.load(str(final_model_path), map_location=device)
    model.load_state_dict(ckpt.get('params_ema', ckpt.get('params', ckpt)))

    evaluator = XyetherEvaluator(device=device)
    report = evaluator.evaluate_model(model)
    display(HTML(evaluator.format_html_report(report, iteration="FINAL")))
else:
    print_status(f"Final model not found at {final_model_path}. Please check Phase 2 output.", "#FF9800")